In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F
#spark=SparkSession.builder.appName('ecommerce')\
    #.config("spark.sql.adaptive.enabled", "true")\
    #.getOrCreate()

catalog_name='ecommerce'
raw_data_path="/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv"

data_schema=StructType([
       StructField("dt",                 StringType(), True),
    StructField("order_ts",           StringType(), True),
    StructField("customer_id",        StringType(), True),
    StructField("order_id",           StringType(), True),
    StructField("item_seq",           StringType(), True),
    StructField("product_id",         StringType(), True),
    StructField("quantity",           StringType(), True),
    StructField("unit_price_currency",StringType(), True),
    StructField("unit_price",         StringType(), True),
    StructField("discount_pct",       StringType(), True),
    StructField("tax_amount",         StringType(), True),
    StructField("channel",            StringType(), True),
    StructField("coupon_code",        StringType(), True),
]) 

df_bronze=spark.read\
    .option('header',True)\
    .schema(data_schema)\
    .option('delimeter',',')\
    .csv(raw_data_path)\
    .withColumn('file_path',F.col('_metadata.file_path'))\
    .withColumn('ingest_time',F.current_timestamp())
df_bronze.write.format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema',True)\
    .saveAsTable(f"{catalog_name}.bronze.brz_order_items")
df_bronze.count()